# Week 87, Day 5: First vertical slice

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w087d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('waymark_domain.py', 'w', encoding='utf-8') as f:
    f.write('import heapq\n\n\nclass UnknownPlace(LookupError):\n    pass\n\n\nclass NoRoute(Exception):\n    pass\n\n\ndef naismith_minutes(km, climb_m):\n    return round(km / 5 * 60 + climb_m / 600 * 60)\n\n\nclass Route:\n    def __init__(self, stops, km, climb_m):\n        self.stops, self.km, self.climb_m = list(stops), km, climb_m\n\n    @property\n    def minutes(self):\n        return naismith_minutes(self.km, self.climb_m)\n\n    def __repr__(self):\n        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"\n\n\nclass PathNetwork:\n    def __init__(self, heights):\n        self.heights = dict(heights)\n        self.paths = {name: {} for name in self.heights}\n\n    def _check(self, *names):\n        for name in names:\n            if name not in self.heights:\n                raise UnknownPlace(name)\n\n    def add_path(self, a, b, km):\n        self._check(a, b)\n        if km <= 0:\n            raise ValueError("km must be positive")\n        self.paths[a][b] = km\n        self.paths[b][a] = km\n\n    def climb(self, stops):\n        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))\n\n    def shortest(self, start, end):\n        self._check(start, end)\n        dist, prev, done = {start: 0}, {}, set()\n        heap = [(0, start)]\n        while heap:\n            d, node = heapq.heappop(heap)\n            if node in done:\n                continue\n            done.add(node)\n            if node == end:\n                break\n            for nxt, km in sorted(self.paths[node].items()):\n                if d + km < dist.get(nxt, float("inf")):\n                    dist[nxt] = d + km\n                    prev[nxt] = node\n                    heapq.heappush(heap, (d + km, nxt))\n        if end not in done:\n            raise NoRoute(f"no route from {start} to {end}")\n        stops = [end]\n        while stops[-1] != start:\n            stops.append(prev[stops[-1]])\n        stops.reverse()\n        return Route(stops, round(dist[end], 2), self.climb(stops))\n')

**Predict before you run:** what will this print?



In [ ]:
from waymark_domain import PathNetwork, NoRoute

network = PathNetwork({"A": 10, "B": 70, "C": 40, "D": 0})
network.add_path("A", "B", 2)
network.add_path("B", "C", 3)
network.add_path("A", "C", 6)
route = network.shortest("A", "C")
print(route.stops, route.km, route.climb_m, route.minutes)
back = network.shortest("C", "A")
print(back.stops, back.climb_m, back.minutes)
try:
    network.shortest("A", "D")
except NoRoute as e:
    print(e)

> **Key idea.** Build in vertical slices: one story through every layer, finished properly, before the next; choose a first slice that's central, thin and touches every layer. Build from the inside out: a pure domain with its own exceptions and invariants, storage that maps it to tables, a service per user action returning plain data, and interfaces that only parse, call and translate outcomes (exit codes for the CLI, status codes for the API). Test the domain with many unit tests written from the scenarios, storage with integration tests on a real in-memory database, and the interfaces with a few end-to-end tests. Tackle the riskiest part early.

### Exercise: The domain, from the spec

Without looking at `waymark_domain.py`, write the domain yourself: `UnknownPlace` (a `LookupError`) and `NoRoute` (an `Exception`); `naismith_minutes(km, climb_m)`; `Route(stops, km, climb_m)` with a `minutes` property; and `PathNetwork(heights)` with `add_path(a, b, km)` (both directions; `UnknownPlace(name)` for an unknown junction, `ValueError("km must be positive")`), `climb(stops)` (the sum of uphill steps) and `shortest(start, end)` (Dijkstra by distance, visiting neighbours in name order; `UnknownPlace(name)` for an unknown start or end; `NoRoute("no route from A to B")`; the route's `km` rounded to 2 places).

In [ ]:
import heapq

class UnknownPlace(LookupError):
    pass

class NoRoute(Exception):
    pass

def naismith_minutes(km, climb_m):
    ___

class Route:
    ___

class PathNetwork:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("HEIGHTS = {\"Mill\": 40, \"Weir\": 55, \"Church\": 80, \"Ridge\": 310, \"Farm\": 120, \"Wood\": 150, \"Lake\": 60, \"Island\": 5}\nPATHS = [(\"Mill\", \"Weir\", 1.5), (\"Weir\", \"Church\", 2.0), (\"Mill\", \"Church\", 4.0), (\"Church\", \"Ridge\", 3.0),\n         (\"Weir\", \"Farm\", 2.5), (\"Farm\", \"Ridge\", 2.0), (\"Farm\", \"Wood\", 1.5), (\"Wood\", \"Lake\", 2.0), (\"Lake\", \"Mill\", 3.5)]\n\ndef _network():\n    n = PathNetwork(HEIGHTS)\n    for a, b, km in PATHS:\n        n.add_path(a, b, km)\n    return n\n\ndef test_routes():\n    \"S1 and S2 on the sample map\"\n    n = _network()\n    r = n.shortest(\"Mill\", \"Ridge\")\n    assert (r.stops, r.km, r.climb_m, r.minutes) == ([\"Mill\", \"Weir\", \"Farm\", \"Ridge\"], 6.0, 270, 99)\n    back = n.shortest(\"Ridge\", \"Mill\")\n    assert (back.stops, back.climb_m, back.minutes) == ([\"Ridge\", \"Farm\", \"Weir\", \"Mill\"], 0, 72)\n    assert n.shortest(\"Lake\", \"Church\").stops == [\"Lake\", \"Mill\", \"Weir\", \"Church\"]\n    assert n.shortest(\"Mill\", \"Mill\").stops == [\"Mill\"] and naismith_minutes(10, 600) == 180\n\ndef test_errors():\n    \"the domain's own exceptions\"\n    n = _network()\n    for call, kind, message in ((lambda: n.shortest(\"Mill\", \"Island\"), NoRoute, \"no route from Mill to Island\"),\n                                (lambda: n.shortest(\"Moon\", \"Mill\"), UnknownPlace, \"Moon\"),\n                                (lambda: n.add_path(\"Mill\", \"Moon\", 1), UnknownPlace, \"Moon\"),\n                                (lambda: n.add_path(\"Mill\", \"Lake\", 0), ValueError, \"km must be positive\")):\n        try:\n            call()\n        except kind as e:\n            assert e.args[0] == message, e\n        else:\n            raise AssertionError(message)\n    assert issubclass(UnknownPlace, LookupError)")

### Exercise: Tests from the scenarios

Write pytest tests for `waymark_domain.py` (shown above), named after their stories (`test_s1_...`, `test_s2_...`), that pass on it and catch all six broken versions. Build small networks in the tests (or a fixture), like the scenarios in Day 2's spec.

In [ ]:
import pytest
from waymark_domain import PathNetwork, NoRoute, UnknownPlace, naismith_minutes

def test_s1_shortest_route():
    n = PathNetwork({"A": 0, "B": 0, "C": 0})
    n.add_path("A", "B", 2)
    n.add_path("B", "C", 3)
    assert n.shortest("A", "C").stops == ["A", "B", "C"]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"waymark_domain.py\").read()\nBROKEN = {\n    \"climbs both ways\": GOOD.replace(\"max(0, self.heights[b] - self.heights[a])\", \"abs(self.heights[b] - self.heights[a])\"),\n    \"fewest stops\": GOOD.replace(\"d + km\", \"d + 1\"),\n    \"climb too costly\": GOOD.replace(\"climb_m / 600 * 60\", \"climb_m / 500 * 60\"),\n    \"no route is None\": GOOD.replace('        if end not in done:\\n            raise NoRoute(f\"no route from {start} to {end}\")', \"        if end not in done:\\n            return None\"),\n    \"unknown places allowed\": GOOD.replace(\"            if name not in self.heights:\\n                raise UnknownPlace(name)\", \"            pass\"),\n    \"zero-length paths\": GOOD.replace('        if km <= 0:\\n            raise ValueError(\"km must be positive\")\\n', \"\"),\n}\ndef test_suite():\n    \"all six caught, tests named by story\"\n    for name, code in BROKEN.items():\n        assert code != GOOD, name\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"], r[\"output\"][-600:]\n    assert all(t.split(\"[\")[0].split(\"::\")[-1].startswith((\"test_s1_\", \"test_s2_\", \"test_s3_\")) for t in r[\"passed\"]), r[\"passed\"]\n    results = against(source, \"waymark_domain\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"waymark_domain.py\", GOOD)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

### Exercise: Storage

Using the given domain, write `save_network(conn, heights, paths)` (in one transaction, insert every junction and every path, a list of `(start, finish, km)`) and `load_network(conn)` (rebuild a `PathNetwork` from the tables, adding paths in the order they were stored). `connect()` (given) creates the schema with foreign keys on.

In [ ]:
import heapq


class UnknownPlace(LookupError):
    pass


class NoRoute(Exception):
    pass


def naismith_minutes(km, climb_m):
    return round(km / 5 * 60 + climb_m / 600 * 60)


class Route:
    def __init__(self, stops, km, climb_m):
        self.stops, self.km, self.climb_m = list(stops), km, climb_m

    @property
    def minutes(self):
        return naismith_minutes(self.km, self.climb_m)

    def __repr__(self):
        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"


class PathNetwork:
    def __init__(self, heights):
        self.heights = dict(heights)
        self.paths = {name: {} for name in self.heights}

    def _check(self, *names):
        for name in names:
            if name not in self.heights:
                raise UnknownPlace(name)

    def add_path(self, a, b, km):
        self._check(a, b)
        if km <= 0:
            raise ValueError("km must be positive")
        self.paths[a][b] = km
        self.paths[b][a] = km

    def climb(self, stops):
        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))

    def shortest(self, start, end):
        self._check(start, end)
        dist, prev, done = {start: 0}, {}, set()
        heap = [(0, start)]
        while heap:
            d, node = heapq.heappop(heap)
            if node in done:
                continue
            done.add(node)
            if node == end:
                break
            for nxt, km in sorted(self.paths[node].items()):
                if d + km < dist.get(nxt, float("inf")):
                    dist[nxt] = d + km
                    prev[nxt] = node
                    heapq.heappush(heap, (d + km, nxt))
        if end not in done:
            raise NoRoute(f"no route from {start} to {end}")
        stops = [end]
        while stops[-1] != start:
            stops.append(prev[stops[-1]])
        stops.reverse()
        return Route(stops, round(dist[end], 2), self.climb(stops))

import sqlite3

SCHEMA = """
CREATE TABLE IF NOT EXISTS junction (name TEXT PRIMARY KEY, height_m INTEGER NOT NULL);
CREATE TABLE IF NOT EXISTS path (start TEXT NOT NULL REFERENCES junction(name),
                                 finish TEXT NOT NULL REFERENCES junction(name),
                                 km REAL NOT NULL CHECK (km > 0));
"""

def connect(path=":memory:"):
    conn = sqlite3.connect(path)
    conn.execute("PRAGMA foreign_keys = ON")
    conn.executescript(SCHEMA)
    return conn

def save_network(conn, heights, paths):
    ___

def load_network(conn):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import sqlite3\n\nHEIGHTS = {\"Mill\": 40, \"Weir\": 55, \"Church\": 80, \"Ridge\": 310, \"Farm\": 120, \"Wood\": 150, \"Lake\": 60, \"Island\": 5}\nPATHS = [(\"Mill\", \"Weir\", 1.5), (\"Weir\", \"Church\", 2.0), (\"Mill\", \"Church\", 4.0), (\"Church\", \"Ridge\", 3.0),\n         (\"Weir\", \"Farm\", 2.5), (\"Farm\", \"Ridge\", 2.0), (\"Farm\", \"Wood\", 1.5), (\"Wood\", \"Lake\", 2.0), (\"Lake\", \"Mill\", 3.5)]\n\ndef test_round_trip():\n    \"what's saved loads back\"\n    conn = connect()\n    save_network(conn, HEIGHTS, PATHS)\n    n = load_network(conn)\n    assert n.heights == HEIGHTS and n.shortest(\"Mill\", \"Ridge\").stops == [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"]\n    assert conn.execute(\"SELECT COUNT(*) FROM path\").fetchone() == (9,)\n\ndef test_constraints():\n    \"bad data never lands\"\n    conn = connect()\n    for paths in ([(\"Mill\", \"Moon\", 1.0)], [(\"Mill\", \"Weir\", 1.0), (\"Weir\", \"Church\", -2.0)]):\n        try:\n            save_network(conn, {\"Mill\": 40, \"Weir\": 55, \"Church\": 80}, paths)\n        except sqlite3.IntegrityError:\n            pass\n        else:\n            raise AssertionError(paths)\n        assert conn.execute(\"SELECT COUNT(*) FROM junction\").fetchone() == (0,), \"all or nothing\"")

### Exercise: The service

With the given domain and storage, write `plan(conn, start, end)`: load the network, find the shortest route, and return a dict with `stops`, `km` (rounded to 1 place), `climb_m` and `minutes`. Let the domain's exceptions pass through: deciding how to report them is the interfaces' job.

In [ ]:
import heapq


class UnknownPlace(LookupError):
    pass


class NoRoute(Exception):
    pass


def naismith_minutes(km, climb_m):
    return round(km / 5 * 60 + climb_m / 600 * 60)


class Route:
    def __init__(self, stops, km, climb_m):
        self.stops, self.km, self.climb_m = list(stops), km, climb_m

    @property
    def minutes(self):
        return naismith_minutes(self.km, self.climb_m)

    def __repr__(self):
        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"


class PathNetwork:
    def __init__(self, heights):
        self.heights = dict(heights)
        self.paths = {name: {} for name in self.heights}

    def _check(self, *names):
        for name in names:
            if name not in self.heights:
                raise UnknownPlace(name)

    def add_path(self, a, b, km):
        self._check(a, b)
        if km <= 0:
            raise ValueError("km must be positive")
        self.paths[a][b] = km
        self.paths[b][a] = km

    def climb(self, stops):
        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))

    def shortest(self, start, end):
        self._check(start, end)
        dist, prev, done = {start: 0}, {}, set()
        heap = [(0, start)]
        while heap:
            d, node = heapq.heappop(heap)
            if node in done:
                continue
            done.add(node)
            if node == end:
                break
            for nxt, km in sorted(self.paths[node].items()):
                if d + km < dist.get(nxt, float("inf")):
                    dist[nxt] = d + km
                    prev[nxt] = node
                    heapq.heappush(heap, (d + km, nxt))
        if end not in done:
            raise NoRoute(f"no route from {start} to {end}")
        stops = [end]
        while stops[-1] != start:
            stops.append(prev[stops[-1]])
        stops.reverse()
        return Route(stops, round(dist[end], 2), self.climb(stops))

import sqlite3

SCHEMA = """
CREATE TABLE IF NOT EXISTS junction (name TEXT PRIMARY KEY, height_m INTEGER NOT NULL);
CREATE TABLE IF NOT EXISTS path (start TEXT NOT NULL REFERENCES junction(name),
                                 finish TEXT NOT NULL REFERENCES junction(name),
                                 km REAL NOT NULL CHECK (km > 0));
"""

def connect(path=":memory:"):
    conn = sqlite3.connect(path)
    conn.execute("PRAGMA foreign_keys = ON")
    conn.executescript(SCHEMA)
    return conn

def save_network(conn, heights, paths):
    with conn:
        conn.executemany("INSERT INTO junction VALUES (?, ?)", list(heights.items()))
        conn.executemany("INSERT INTO path VALUES (?, ?, ?)", paths)

def load_network(conn):
    network = PathNetwork(dict(conn.execute("SELECT name, height_m FROM junction")))
    for start, finish, km in conn.execute("SELECT start, finish, km FROM path ORDER BY rowid"):
        network.add_path(start, finish, km)
    return network

def plan(conn, start, end):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("HEIGHTS = {\"Mill\": 40, \"Weir\": 55, \"Church\": 80, \"Ridge\": 310, \"Farm\": 120, \"Wood\": 150, \"Lake\": 60, \"Island\": 5}\nPATHS = [(\"Mill\", \"Weir\", 1.5), (\"Weir\", \"Church\", 2.0), (\"Mill\", \"Church\", 4.0), (\"Church\", \"Ridge\", 3.0),\n         (\"Weir\", \"Farm\", 2.5), (\"Farm\", \"Ridge\", 2.0), (\"Farm\", \"Wood\", 1.5), (\"Wood\", \"Lake\", 2.0), (\"Lake\", \"Mill\", 3.5)]\n\ndef test_service():\n    \"plain data out, domain errors through\"\n    conn = connect()\n    save_network(conn, HEIGHTS, PATHS)\n    assert plan(conn, \"Mill\", \"Ridge\") == {\"stops\": [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"], \"km\": 6.0, \"climb_m\": 270, \"minutes\": 99}\n    assert plan(conn, \"Mill\", \"Wood\") == {\"stops\": [\"Mill\", \"Lake\", \"Wood\"], \"km\": 5.5, \"climb_m\": 110, \"minutes\": 77}\n    for args, kind in (((\"Mill\", \"Island\"), NoRoute), ((\"Moon\", \"Mill\"), UnknownPlace)):\n        try:\n            plan(conn, *args)\n        except kind:\n            pass\n        else:\n            raise AssertionError(args)")

### Exercise: The command-line tool

Write `main(argv, conn, out)` for `waymark plan FROM TO`: use `argparse` (a `plan` subcommand with two positional arguments; on invalid arguments argparse raises `SystemExit`: return 2). On success write `Mill -> Weir -> Farm -> Ridge: 6.0 km, 1 h 39 min` (minutes as two digits) and a newline to `out`, and return 0. For an unknown place write `unknown place: NAME` and return 2; for no route write the error's message (`no route from A to B`) and return 1.

In [ ]:
import argparse
import contextlib
import io

import heapq


class UnknownPlace(LookupError):
    pass


class NoRoute(Exception):
    pass


def naismith_minutes(km, climb_m):
    return round(km / 5 * 60 + climb_m / 600 * 60)


class Route:
    def __init__(self, stops, km, climb_m):
        self.stops, self.km, self.climb_m = list(stops), km, climb_m

    @property
    def minutes(self):
        return naismith_minutes(self.km, self.climb_m)

    def __repr__(self):
        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"


class PathNetwork:
    def __init__(self, heights):
        self.heights = dict(heights)
        self.paths = {name: {} for name in self.heights}

    def _check(self, *names):
        for name in names:
            if name not in self.heights:
                raise UnknownPlace(name)

    def add_path(self, a, b, km):
        self._check(a, b)
        if km <= 0:
            raise ValueError("km must be positive")
        self.paths[a][b] = km
        self.paths[b][a] = km

    def climb(self, stops):
        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))

    def shortest(self, start, end):
        self._check(start, end)
        dist, prev, done = {start: 0}, {}, set()
        heap = [(0, start)]
        while heap:
            d, node = heapq.heappop(heap)
            if node in done:
                continue
            done.add(node)
            if node == end:
                break
            for nxt, km in sorted(self.paths[node].items()):
                if d + km < dist.get(nxt, float("inf")):
                    dist[nxt] = d + km
                    prev[nxt] = node
                    heapq.heappush(heap, (d + km, nxt))
        if end not in done:
            raise NoRoute(f"no route from {start} to {end}")
        stops = [end]
        while stops[-1] != start:
            stops.append(prev[stops[-1]])
        stops.reverse()
        return Route(stops, round(dist[end], 2), self.climb(stops))

import sqlite3

SCHEMA = """
CREATE TABLE IF NOT EXISTS junction (name TEXT PRIMARY KEY, height_m INTEGER NOT NULL);
CREATE TABLE IF NOT EXISTS path (start TEXT NOT NULL REFERENCES junction(name),
                                 finish TEXT NOT NULL REFERENCES junction(name),
                                 km REAL NOT NULL CHECK (km > 0));
"""

def connect(path=":memory:"):
    conn = sqlite3.connect(path)
    conn.execute("PRAGMA foreign_keys = ON")
    conn.executescript(SCHEMA)
    return conn

def save_network(conn, heights, paths):
    with conn:
        conn.executemany("INSERT INTO junction VALUES (?, ?)", list(heights.items()))
        conn.executemany("INSERT INTO path VALUES (?, ?, ?)", paths)

def load_network(conn):
    network = PathNetwork(dict(conn.execute("SELECT name, height_m FROM junction")))
    for start, finish, km in conn.execute("SELECT start, finish, km FROM path ORDER BY rowid"):
        network.add_path(start, finish, km)
    return network

def plan(conn, start, end):
    route = load_network(conn).shortest(start, end)
    return {"stops": route.stops, "km": round(route.km, 1), "climb_m": route.climb_m, "minutes": route.minutes}

def main(argv, conn, out):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import io\n\nHEIGHTS = {\"Mill\": 40, \"Weir\": 55, \"Church\": 80, \"Ridge\": 310, \"Farm\": 120, \"Wood\": 150, \"Lake\": 60, \"Island\": 5}\nPATHS = [(\"Mill\", \"Weir\", 1.5), (\"Weir\", \"Church\", 2.0), (\"Mill\", \"Church\", 4.0), (\"Church\", \"Ridge\", 3.0),\n         (\"Weir\", \"Farm\", 2.5), (\"Farm\", \"Ridge\", 2.0), (\"Farm\", \"Wood\", 1.5), (\"Wood\", \"Lake\", 2.0), (\"Lake\", \"Mill\", 3.5)]\n\ndef _run(argv):\n    conn = connect()\n    save_network(conn, HEIGHTS, PATHS)\n    out = io.StringIO()\n    code = main(argv, conn, out)\n    return code, out.getvalue()\n\ndef test_cli():\n    \"output and exit codes\"\n    assert _run([\"plan\", \"Mill\", \"Ridge\"]) == (0, \"Mill -> Weir -> Farm -> Ridge: 6.0 km, 1 h 39 min\\n\")\n    assert _run([\"plan\", \"Ridge\", \"Mill\"]) == (0, \"Ridge -> Farm -> Weir -> Mill: 6.0 km, 1 h 12 min\\n\")\n    assert _run([\"plan\", \"Mill\", \"Moon\"]) == (2, \"unknown place: Moon\\n\")\n    assert _run([\"plan\", \"Mill\", \"Island\"]) == (1, \"no route from Mill to Island\\n\")\n    assert _run([\"plan\", \"Mill\"])[0] == 2 and _run([])[0] == 2")

### Exercise: The API

Write `handle(method, target, conn)` for `GET /routes?from=A&to=B`: return `(200, plan_result)`; `(400, {"error": "from and to are required"})` if either parameter is missing or empty; `(404, {"error": "unknown place: NAME"})` and `(404, {"error": "no route from A to B"})` for the domain's errors; `(405, {"error": "method not allowed"})` for another method on `/routes`; and `(404, {"error": "not found"})` for any other path.

In [ ]:
from urllib.parse import parse_qs, urlsplit

import heapq


class UnknownPlace(LookupError):
    pass


class NoRoute(Exception):
    pass


def naismith_minutes(km, climb_m):
    return round(km / 5 * 60 + climb_m / 600 * 60)


class Route:
    def __init__(self, stops, km, climb_m):
        self.stops, self.km, self.climb_m = list(stops), km, climb_m

    @property
    def minutes(self):
        return naismith_minutes(self.km, self.climb_m)

    def __repr__(self):
        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"


class PathNetwork:
    def __init__(self, heights):
        self.heights = dict(heights)
        self.paths = {name: {} for name in self.heights}

    def _check(self, *names):
        for name in names:
            if name not in self.heights:
                raise UnknownPlace(name)

    def add_path(self, a, b, km):
        self._check(a, b)
        if km <= 0:
            raise ValueError("km must be positive")
        self.paths[a][b] = km
        self.paths[b][a] = km

    def climb(self, stops):
        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))

    def shortest(self, start, end):
        self._check(start, end)
        dist, prev, done = {start: 0}, {}, set()
        heap = [(0, start)]
        while heap:
            d, node = heapq.heappop(heap)
            if node in done:
                continue
            done.add(node)
            if node == end:
                break
            for nxt, km in sorted(self.paths[node].items()):
                if d + km < dist.get(nxt, float("inf")):
                    dist[nxt] = d + km
                    prev[nxt] = node
                    heapq.heappush(heap, (d + km, nxt))
        if end not in done:
            raise NoRoute(f"no route from {start} to {end}")
        stops = [end]
        while stops[-1] != start:
            stops.append(prev[stops[-1]])
        stops.reverse()
        return Route(stops, round(dist[end], 2), self.climb(stops))

import sqlite3

SCHEMA = """
CREATE TABLE IF NOT EXISTS junction (name TEXT PRIMARY KEY, height_m INTEGER NOT NULL);
CREATE TABLE IF NOT EXISTS path (start TEXT NOT NULL REFERENCES junction(name),
                                 finish TEXT NOT NULL REFERENCES junction(name),
                                 km REAL NOT NULL CHECK (km > 0));
"""

def connect(path=":memory:"):
    conn = sqlite3.connect(path)
    conn.execute("PRAGMA foreign_keys = ON")
    conn.executescript(SCHEMA)
    return conn

def save_network(conn, heights, paths):
    with conn:
        conn.executemany("INSERT INTO junction VALUES (?, ?)", list(heights.items()))
        conn.executemany("INSERT INTO path VALUES (?, ?, ?)", paths)

def load_network(conn):
    network = PathNetwork(dict(conn.execute("SELECT name, height_m FROM junction")))
    for start, finish, km in conn.execute("SELECT start, finish, km FROM path ORDER BY rowid"):
        network.add_path(start, finish, km)
    return network

def plan(conn, start, end):
    route = load_network(conn).shortest(start, end)
    return {"stops": route.stops, "km": round(route.km, 1), "climb_m": route.climb_m, "minutes": route.minutes}

def handle(method, target, conn):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("HEIGHTS = {\"Mill\": 40, \"Weir\": 55, \"Church\": 80, \"Ridge\": 310, \"Farm\": 120, \"Wood\": 150, \"Lake\": 60, \"Island\": 5}\nPATHS = [(\"Mill\", \"Weir\", 1.5), (\"Weir\", \"Church\", 2.0), (\"Mill\", \"Church\", 4.0), (\"Church\", \"Ridge\", 3.0),\n         (\"Weir\", \"Farm\", 2.5), (\"Farm\", \"Ridge\", 2.0), (\"Farm\", \"Wood\", 1.5), (\"Wood\", \"Lake\", 2.0), (\"Lake\", \"Mill\", 3.5)]\n\ndef test_api():\n    \"status codes for every outcome\"\n    conn = connect()\n    save_network(conn, HEIGHTS, PATHS)\n    assert handle(\"GET\", \"/routes?from=Mill&to=Ridge\", conn) == (200, {\"stops\": [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"], \"km\": 6.0, \"climb_m\": 270, \"minutes\": 99})\n    assert handle(\"GET\", \"/routes?from=Mill\", conn) == (400, {\"error\": \"from and to are required\"})\n    assert handle(\"GET\", \"/routes?from=&to=Mill\", conn)[0] == 400\n    assert handle(\"GET\", \"/routes?from=Mill&to=Moon\", conn) == (404, {\"error\": \"unknown place: Moon\"})\n    assert handle(\"GET\", \"/routes?from=Island&to=Mill\", conn) == (404, {\"error\": \"no route from Island to Mill\"})\n    assert handle(\"POST\", \"/routes\", conn) == (405, {\"error\": \"method not allowed\"})\n    assert handle(\"GET\", \"/walks\", conn) == (404, {\"error\": \"not found\"})")

### Bug hunt: Finished too soon

This Dijkstra marks a junction finished when it's first **pushed**, not when it's **popped**, so A's direct path to C (6 km) wins over the shorter way through B: it prints `['A', 'C'] 6`. Mark junctions done when they come off the heap, so it prints `['A', 'B', 'C'] 5`.

In [ ]:
import heapq

paths = {"A": {"B": 2, "C": 6}, "B": {"A": 2, "C": 3}, "C": {"A": 6, "B": 3}}
dist, prev, done = {"A": 0}, {}, {"A"}
heap = [(0, "A")]
while heap:
    d, node = heapq.heappop(heap)
    for nxt, km in sorted(paths[node].items()):
        if nxt not in done:
            done.add(nxt)
            dist[nxt], prev[nxt] = d + km, node
            heapq.heappush(heap, (d + km, nxt))
stops = ["C"]
while stops[-1] != "A":
    stops.append(prev[stops[-1]])
print(stops[::-1], dist["C"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"done when popped\"\n    expect_output(\"['A', 'B', 'C'] 5\")")

### Bug hunt: Climbing downhill

The climb counts every change in height, so walking down from the ridge counts as climbing: it prints `climb: 270`. Count only uphill steps, so it prints `climb: 0`.

In [ ]:
heights = {"Ridge": 310, "Farm": 120, "Weir": 55, "Mill": 40}
stops = ["Ridge", "Farm", "Weir", "Mill"]
climb = sum(abs(heights[b] - heights[a]) for a, b in zip(stops, stops[1:]))
print("climb:", climb)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"uphill only\"\n    expect_output(\"climb: 0\")")

### Bug hunt: A failure that looked like success

The command prints its error but still returns 0, so a script calling it can't tell it failed: it prints `exit code: 0`. Return 1 for no route, so it prints `exit code: 1`.

In [ ]:
class NoRoute(Exception):
    pass

def plan(start, end):
    raise NoRoute(f"no route from {start} to {end}")

def main(argv):
    try:
        plan(*argv)
    except NoRoute as e:
        print(e)
    return 0

print("exit code:", main(["Mill", "Island"]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"non-zero on failure\"\n    expect_output(\"no route from Mill to Island\\nexit code: 1\")")

### Puzzle: Through every layer

Arrange the calls of one request through Waymark's layers, from the outside in and back out. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # Naismith minutes for 6 km with 270 m of climbing
q = ___   # the climb from height 120 to 310 and back down to 55, in metres
r = ___   # "unit", "integration" or "end-to-end": a test of save_network on an in-memory SQLite database
s = ___   # the API status for a request whose "to" names no junction
t = ___   # True or False: the domain should know about HTTP status codes

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 99 and q == 190 and r == \"integration\", \"p, q and r\"\n    assert s == 404 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: round walks near a target time (S4)

The riskiest story is S4: a **round walk** from a start, close to a target time. Write `round_walks(network, start, target, tolerance=15, max_stops=8)` using the given domain. A round walk starts and ends at `start`, visits at least two other junctions, and never visits a junction twice (except returning to the start); it has at most `max_stops` junctions before the return. Explore by depth-first search, trying neighbours in name order. A loop and its reverse are the same walk: keep the direction with **less climb** (ties: the direction whose second stop comes first alphabetically). Return the `Route`s (with `km` rounded to 2 places) whose minutes are within `tolerance` of `target`, closest first (ties by km, then by stops). Raise `UnknownPlace` for an unknown start.

In [ ]:
import heapq


class UnknownPlace(LookupError):
    pass


class NoRoute(Exception):
    pass


def naismith_minutes(km, climb_m):
    return round(km / 5 * 60 + climb_m / 600 * 60)


class Route:
    def __init__(self, stops, km, climb_m):
        self.stops, self.km, self.climb_m = list(stops), km, climb_m

    @property
    def minutes(self):
        return naismith_minutes(self.km, self.climb_m)

    def __repr__(self):
        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"


class PathNetwork:
    def __init__(self, heights):
        self.heights = dict(heights)
        self.paths = {name: {} for name in self.heights}

    def _check(self, *names):
        for name in names:
            if name not in self.heights:
                raise UnknownPlace(name)

    def add_path(self, a, b, km):
        self._check(a, b)
        if km <= 0:
            raise ValueError("km must be positive")
        self.paths[a][b] = km
        self.paths[b][a] = km

    def climb(self, stops):
        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))

    def shortest(self, start, end):
        self._check(start, end)
        dist, prev, done = {start: 0}, {}, set()
        heap = [(0, start)]
        while heap:
            d, node = heapq.heappop(heap)
            if node in done:
                continue
            done.add(node)
            if node == end:
                break
            for nxt, km in sorted(self.paths[node].items()):
                if d + km < dist.get(nxt, float("inf")):
                    dist[nxt] = d + km
                    prev[nxt] = node
                    heapq.heappush(heap, (d + km, nxt))
        if end not in done:
            raise NoRoute(f"no route from {start} to {end}")
        stops = [end]
        while stops[-1] != start:
            stops.append(prev[stops[-1]])
        stops.reverse()
        return Route(stops, round(dist[end], 2), self.climb(stops))

def round_walks(network, start, target, tolerance=15, max_stops=8):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("HEIGHTS = {\"Mill\": 40, \"Weir\": 55, \"Church\": 80, \"Ridge\": 310, \"Farm\": 120, \"Wood\": 150, \"Lake\": 60, \"Island\": 5}\nPATHS = [(\"Mill\", \"Weir\", 1.5), (\"Weir\", \"Church\", 2.0), (\"Mill\", \"Church\", 4.0), (\"Church\", \"Ridge\", 3.0),\n         (\"Weir\", \"Farm\", 2.5), (\"Farm\", \"Ridge\", 2.0), (\"Farm\", \"Wood\", 1.5), (\"Wood\", \"Lake\", 2.0), (\"Lake\", \"Mill\", 3.5)]\n\ndef _network():\n    n = PathNetwork(HEIGHTS)\n    for a, b, km in PATHS:\n        n.add_path(a, b, km)\n    return n\n\ndef test_round_walks():\n    \"loops, deduplicated, closest first\"\n    n = _network()\n    walks = round_walks(n, \"Mill\", 140)\n    assert [(w.stops, w.km, w.climb_m, w.minutes) for w in walks] == [([\"Mill\", \"Lake\", \"Wood\", \"Farm\", \"Weir\", \"Mill\"], 11.0, 110, 143)]\n    walks = round_walks(n, \"Mill\", 200, tolerance=20)\n    assert [w.minutes for w in walks] == [200, 216, 183]\n    assert walks[0].stops == [\"Mill\", \"Church\", \"Weir\", \"Farm\", \"Wood\", \"Lake\", \"Mill\"]\n    short = round_walks(n, \"Mill\", 94, tolerance=0)\n    assert [w.stops for w in short] == [[\"Mill\", \"Church\", \"Weir\", \"Mill\"]], \"the tie goes to Church before Weir\"\n\ndef test_limits():\n    \"max stops, no loops, unknown start\"\n    n = _network()\n    assert len(round_walks(n, \"Mill\", 0, tolerance=1000)) == 6\n    assert [w.minutes for w in round_walks(n, \"Mill\", 0, tolerance=1000, max_stops=3)] == [94]\n    assert round_walks(n, \"Island\", 60, tolerance=1000) == []\n    try:\n        round_walks(n, \"Moon\", 60)\n    except UnknownPlace:\n        pass\n    else:\n        raise AssertionError(\"unknown start\")")

> **What did the spike tell you?.** On a map of eight junctions, the depth-first search is instant. On a real map of 5,000 junctions it would explore an astronomical number of paths: the number of simple loops grows exponentially. That's exactly the kind of thing to learn in week one. The options for Waymark: limit `max_stops` (simple, but long walks need many junctions), search only junctions within half the target distance of the start (pruning by distance), or build loops from two shortest routes to a turning point (fast, good enough). Write an ADR for the choice. What's the riskiest part of **your** capstone, and what would a spike for it look like?

## Warm-up

That's Waymark's first scenario, running: A to C via B is 5 km, the climb counts only the uphill parts (so the way back is different), and a place with no paths gives a clear error rather than a crash. Open `waymark_domain.py` above: it's the domain layer of today's slice, about 70 lines.

## Slices, not layers

The tempting order is layer by layer: design the whole database, then all the services, then the whole API. It goes wrong in a predictable way: nothing works end to end until week five, problems in how the layers fit together appear late, and if time runs out you have three half-finished layers and nothing a user can try.

A **vertical slice** cuts the other way: one story, through every layer, finished properly (validated, tested, documented) before the next. After the first slice there's something to show; after each one the product is a little more useful; and if time runs out, what's missing are the **last** stories, not the whole thing working. Choose the first slice to be:

- **central**: the story the product is for (for Waymark, finding a route);
- **thin**: the simplest version that's still useful (shortest by distance, no preferences yet);
- **touching every layer**, so the skeleton is proved under real load.

## The domain

Build from the inside out, starting with the part that has no dependencies. Waymark's domain (open the file) has:

- **`PathNetwork`**: junctions with heights, and paths between them with lengths; `add_path` refuses unknown places and non-positive lengths, so an invalid network can't be built.
- **`shortest(start, end)`**: Dijkstra's algorithm (Week 50) on distance, returning a **`Route`**: its stops, kilometres and climb.
- **The climb** of a route counts only uphill steps (`max(0, next height − this height)`), so the same route has a different climb, and time, in each direction.
- **`Route.minutes`** applies Naismith's rule.
- **Its own exceptions**: `UnknownPlace` and `NoRoute`. The domain says **what** went wrong in its own terms; deciding how to report it (an exit code, a 404) is the interfaces' job.

## Storage and the service

**Storage** turns the network into rows and back: a `junction` table and a `path` table, with the constraints from Day 3, and a `load_network(conn)` that rebuilds a `PathNetwork`. It knows SQL; the domain doesn't.

The **service** for S1 and S2 is one function: `plan(conn, start, end)` loads the network, asks the domain for the route, and returns plain data (a dict) for whichever interface called it. Later, when loading the whole map for each request is too slow, only this layer changes (perhaps to cache the network).

## The interfaces

The **command-line tool** and the **API** both call `plan`, and each translates the result into its own language:

| Outcome | CLI | API |
|---|---|---|
| a route | prints `Mill -> Weir -> Ridge: 6.0 km, 1 h 39 min`, exit code 0 | `200` with the route as JSON |
| missing arguments | argparse's usage message, exit code 2 | `400` with an error message |
| an unknown place | `unknown place: Moon`, exit code 2 | `404` with an error message |
| no route | `no route from Mill to Island`, exit code 1 | `404` with an error message |

Nothing in either interface knows about Dijkstra or SQL: they parse, call, and translate.

## Tests at each level

Test each layer at the level where its behaviour lives:

- **Unit tests** for the domain: fast, many, written straight from the scenarios (`test_s1_shortest_route`, `test_s1_no_route`, `test_s2_time_with_climbing`), with small hand-made networks.
- **Integration tests** for storage: a real SQLite database in memory, checking that what's saved loads back the same and that constraints reject bad data.
- **A few end-to-end tests** through each interface: the CLI's output and exit code, the API's status and body. Slower and broader, so fewer.

That shape (many small tests at the bottom, a few broad ones at the top) is the **test pyramid**. Name tests after their stories so tomorrow's review can trace them (Day 2's `trace`).

## Practice

## Quiz

## Daily challenge

## Your capstone today

On your own computer, build your first vertical slice:

1. **Choose the story**: central, thin, every layer. Write it at the top of today's notes.
2. **Build inside out**: domain (with its own exceptions), storage, service, then one interface. Each step on a branch, each commit conventional, each merge with green CI.
3. **Tests from the scenarios**, named after the story: unit tests for the domain, an integration test for storage, one end-to-end test through the interface.
4. **Update the README's usage section** with a real example and its real output.
5. **A spike for your riskiest part**, timeboxed to three hours, ending in an ADR.

## Before you go

Run your slice the way a user would (from a fresh terminal, following the README) and note every rough edge you meet. They're tomorrow's input. Tomorrow is **review and plan**: looking back at the week and planning the first build sprint.